In [ ]:
import torch
import math
from transformers import (
    AutoModelForMaskedLM, 
    AutoTokenizer, 
    DataCollatorForLanguageModeling, 
    TrainingArguments, 
    Trainer,
    EarlyStoppingCallback
)
from datasets import load_dataset

/home/ml/.conda/envs/ykmlen/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
model_id = "DeepChem/ChemBERTa-100M-MLM"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForMaskedLM.from_pretrained(model_id)

In [3]:
dataset = load_dataset("text", data_files="pfas_smiles_unique.txt", split="train")
dataset = dataset.train_test_split(test_size=0.01,seed=42) 

In [4]:
len(dataset["train"]), len(dataset["test"])

(6762222, 68306)

In [5]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=128
    )

tokenized_datasets = dataset.map(
    tokenize_function, 
    batched=True, 
    remove_columns=["text"],
    num_proc=64
)

In [6]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)

In [ ]:
# min_len = 1000
# for item in tokenized_datasets['test']:
#     length = sum(item['attention_mask'])
#     if length < min_len:
#         min_len = length

# print(f"{min_len}")

In [ ]:
training_args = TrainingArguments(
    output_dir="./ChemBERTa-Full_FT-PFAS",
    overwrite_output_dir=False,
    
    per_device_train_batch_size=128,
    num_train_epochs=5,
    learning_rate=1e-5,
    per_device_eval_batch_size=256,
    
    lr_scheduler_type="cosine",   
    warmup_ratio=0.1,            
    
    evaluation_strategy="steps", 
    eval_steps=1000,              
    save_strategy="steps",    
    save_steps=1000,
    logging_steps=1000,           
    load_best_model_at_end=True,  
    
             
    report_to="tensorboard",           
    metric_for_best_model="eval_loss",   
    greater_is_better=False,  
    bf16=True ,  
    dataloader_drop_last=True,         
)

/home/ml/.conda/envs/ykmlen/lib/python3.10/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],

)

train_result = trainer.train()

eval_results = trainer.evaluate()
perplexity = math.exp(eval_results['eval_loss'])
print(f"Final Perplexity: {perplexity:.2f}")
trainer.save_model("./ChemBERTa-Full_FT-PFAS/best_full_finetuned_model")

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


/home/ml/.conda/envs/ykmlen/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss,Validation Loss
1000,1.325800,0.837907
2000,0.740400,0.597209
3000,0.586900,0.506901
4000,0.513500,0.455074
5000,0.470900,0.421330
6000,0.441200,0.398030
7000,0.417100,0.381364
8000,0.396800,0.362645
9000,0.379100,0.347815
10000,0.367500,0.335622


/home/ml/.conda/envs/ykmlen/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/ml/.conda/envs/ykmlen/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/ml/.conda/envs/ykmlen/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/ml/.conda/envs/ykmlen/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/ml/.conda/envs/ykmlen/lib/

Final Perplexity: 1.26
